# Detector-in-the-loop anti-jamming for UAV links (WiFiSpectralJam, real spectral scans)
Kaggle notebook v3 (adds FBR policy and saves detector-score artefacts for offline policy studies; faster: single-thread torch, numpy DQN inference, parallel closed-loop runs, checkpointed outputs). Add the dataset **daniaherzalla/radio-frequency-jamming** as input (Add data -> search "radio-frequency-jamming").
Accelerator: none needed (CPU is enough); torch is only used for the small DQN.
Outputs go to /kaggle/working/out_v3 (CSV saved after every stage). Send the whole out_v3 folder back.

In [ ]:
import os, glob, json, time, math, random, warnings, functools
print=functools.partial(print,flush=True)
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
T0=time.time()
CFG=dict(
    BANDS=['5','2.4'],          # 5 GHz first (harder); '2.4' second
    MAX_BENIGN_PER_LOC=2500,    # files per benign location (passive background) and floor
    MAX_MAL_PER_COND=900,       # files per (target channel, power)
    SEEDS=list(range(10)),      # closed-loop seeds
    T_SLOTS=2000,               # slots per closed-loop episode
    DQN_EPISODES=25,            # DQN training episodes (per band, per variant)
    DQN_T=1000,                 # slots per DQN training episode
    SMOKE=False,                # True only for a tiny local test
    OUT='/kaggle/working/out_v3',
)
def find_root():
    for c in ['/kaggle/input/radio-frequency-jamming','/kaggle/input/datasets/daniaherzalla/radio-frequency-jamming']+glob.glob('/kaggle/input/*'):
        if glob.glob(os.path.join(c,'**','manifest.csv'),recursive=True): return c
    raise FileNotFoundError('Add the dataset daniaherzalla/radio-frequency-jamming as notebook input')
ROOT=os.environ.get('WSJ_ROOT') or find_root()
if os.environ.get('WSJ_SMOKE'): CFG.update(SMOKE=True,MAX_BENIGN_PER_LOC=40,MAX_MAL_PER_COND=12,SEEDS=[0,1],T_SLOTS=300,DQN_EPISODES=3,DQN_T=300,OUT=os.environ.get('WSJ_OUT','out_smoke'))
os.makedirs(CFG['OUT'],exist_ok=True)
MAN=glob.glob(os.path.join(ROOT,'**','manifest.csv'),recursive=True)[0]
print('root',ROOT,'\nmanifest',MAN)

## 1. Manifest and file selection (passive scans; benign background per location, floor, malicious Gaussian-noise)

In [ ]:
man=pd.read_csv(MAN)
print(man.shape); print(man.columns.tolist()); print(man.head(3).T)

In [ ]:
def col(df,*names):
    for n in names:
        if n in df.columns: return n
    raise KeyError(names)
try: C_PATH=col(man,'relative_path','file_path','path','filepath','relpath','rel_path')
except KeyError: C_PATH=[c for c in man.columns if 'path' in c.lower()][0]
C_MODE=col(man,'scan_mode'); C_LAB=col(man,'label'); C_SUB=col(man,'benign_subtype')
C_LOC=col(man,'location'); C_POW=col(man,'power_dbm'); C_CH=col(man,'channel_mhz'); C_BAND=col(man,'band')
m=man[man[C_MODE].astype(str).str.contains('passive',case=False)].copy()
m['lab']=m[C_LAB].astype(str).str.lower().map(lambda s:1 if 'mal' in s else 0)
m['bandn']=m[C_BAND].astype(str).str.extract(r'(2\.4|5)')[0]
miss=m.bandn.isna()
if miss.any():   # fallback: infer band from the file path
    pth=m.loc[miss,C_PATH].astype(str).str.lower()
    m.loc[miss,'bandn']=np.where(pth.str.contains(r'2[._]?4'),'2.4',np.where(pth.str.contains(r'5[._]?g|5ghz'),'5',np.nan))
    print('band inferred from path for',int(miss.sum()),'files; still missing',int(m.bandn.isna().sum()))
num=lambda x: pd.to_numeric(x.astype(str).str.replace('neg','-').str.extract(r'(-?\d+\.?\d*)')[0],errors='coerce')
m['pw']=num(m[C_POW]); m['tch']=num(m[C_CH])
m['sub']=m[C_SUB].astype(str).str.lower(); m['loc']=m[C_LOC].astype(str).str.lower()
# benign group label: floor or background location
m['grp']=np.where(m.lab==1,'mal',np.where(m['sub'].str.contains('floor'),'floor',m['loc']))
print(m.groupby(['bandn','grp']).size())
print(m[m.lab==1].groupby(['bandn','tch','pw']).size())

In [ ]:
rng0=np.random.default_rng(0)
def pick(df,n):
    return df if len(df)<=n else df.sample(n,random_state=0)
sel=[]
for b in CFG['BANDS']:
    mb=m[m.bandn==b]
    for g,dg in mb[mb.lab==0].groupby('grp'): sel.append(pick(dg,CFG['MAX_BENIGN_PER_LOC']))
    for (c,p),dg in mb[mb.lab==1].groupby(['tch','pw']): sel.append(pick(dg,CFG['MAX_MAL_PER_COND']))
S=pd.concat(sel,ignore_index=True)
print('selected files',len(S)); print(S.groupby(['bandn','grp']).size())

## 2. Per-channel spectral features from raw CSV files
For each file and each scanned centre frequency: mean, std, q10, q50, q90, max of 7 driver fields, plus the channel mean minus the across-channel median (relative). Row counts are NOT used (they differ by class and would be a shortcut).

In [ ]:
FIELDS=['rssi','relpwr_db','max_magnitude','base_pwr_db','total_gain_db','avgpwr_db','noise']
STATS=['mean','std','q10','q50','q90','max']
CH={'2.4':[2412,2417,2422,2427,2432,2437,2442,2447,2452,2457,2462],
    '5':[5180,5200,5220,5240,5260,5280,5300,5320,5745,5765,5785,5805,5825]}
def file_tensor(path,band):
    d=pd.read_csv(path)
    d.columns=[c.strip() for c in d.columns]
    chs=CH[band]; out=np.full((len(chs),len(FIELDS)*len(STATS)),np.nan,dtype=np.float32)
    f=d['freq1'].round().astype(int).values
    for i,c in enumerate(chs):
        x=d.loc[f==c,FIELDS].values.astype(np.float32)
        if len(x)==0: continue
        q=np.quantile(x,[0.1,0.5,0.9],axis=0)
        out[i]=np.concatenate([x.mean(0),x.std(0),q[0],q[1],q[2],x.max(0)])
    rel=out[:,:len(FIELDS)]-np.nanmedian(out[:,:len(FIELDS)],axis=0,keepdims=True)
    return np.concatenate([out,rel],axis=1)
FEAT_NAMES=[f'{s}_{f}' for s in STATS for f in FIELDS]+[f'rel_{f}' for f in FIELDS]

In [ ]:
from concurrent.futures import ThreadPoolExecutor
def load_band(b):
    sb=S[S.bandn==b].reset_index(drop=True)
    base=os.path.dirname(os.path.dirname(os.path.dirname(MAN)))
    def full(p):
        for r in [ROOT,base]:
            q=os.path.join(r,p)
            if os.path.exists(q): return q
        hits=glob.glob(os.path.join(ROOT,'**',os.path.basename(p)),recursive=True)
        return hits[0] if hits else None
    paths=[full(p) for p in sb[C_PATH]]
    ok=[i for i,p in enumerate(paths) if p]
    print(b,'GHz: files found',len(ok),'/',len(sb))
    t=time.time()
    with ThreadPoolExecutor(8) as ex: X=list(ex.map(lambda i:file_tensor(paths[i],b),ok))
    print(f'  loaded in {time.time()-t:.0f}s')
    return sb.iloc[ok].reset_index(drop=True), np.stack(X)
DATA={b:load_band(b) for b in CFG['BANDS']}

## 3. Splits and domains (file level)
Train domain: benign = floor + background locations except the hardest (location1), malicious powers +6, +9, +12 dBm.
Shifted domains: **LOC** = unseen benign background (location1); **POW** = unseen weak jammer (+3 dBm); **LOC+POW** both.

In [ ]:
def make_splits(meta,seed=0):
    r=np.random.default_rng(seed); meta=meta.copy()
    u=r.random(len(meta)); meta['part']=np.where(u<0.6,'train',np.where(u<0.75,'cal','test'))
    meta['dom']='train_dom'
    meta.loc[(meta.lab==0)&(meta.grp=='location1'),'dom']='loc_shift'
    meta.loc[(meta.lab==1)&(meta.pw==meta.pw.min()),'dom']='pow_shift'
    meta.loc[meta.dom!='train_dom','part']='test'   # shifted files are never used for training or calibration
    return meta
for b in CFG['BANDS']:
    meta,X=DATA[b]; DATA[b]=(make_splits(meta),X)
    print(b, DATA[b][0].groupby(['dom','part','lab']).size().unstack(fill_value=0))

## 4. Per-channel jamming detector (channel-agnostic: no channel index among inputs)

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
def pairs(meta,X,idx,band):
    chs=CH[band]; K=len(chs); rows=[];y=[];fid=[]
    for i in idx:
        t=meta.tch.iat[i]; lab=meta.lab.iat[i]
        for k in range(K):
            rows.append(X[i,k]); y.append(int(lab==1 and chs[k]==t)); fid.append(i)
    return np.array(rows),np.array(y),np.array(fid)
DET={}; DETRES=[]
for b in CFG['BANDS']:
    meta,X=DATA[b]
    tr=np.where((meta.dom=='train_dom')&(meta.part=='train'))[0]
    Xtr,ytr,_=pairs(meta,X,tr,b)
    models={'HistGB':HistGradientBoostingClassifier(max_iter=300,learning_rate=0.08,random_state=0),
            'LogReg':make_pipeline(SimpleImputer(),StandardScaler(),LogisticRegression(max_iter=2000,C=0.5))}
    for name,mdl in models.items():
        t=time.time(); mdl.fit(Xtr,ytr); fit_s=time.time()-t
        DET[(b,name)]=mdl
        for dname,mask in {'matched':(meta.dom=='train_dom')&(meta.part=='test'),
                           'loc_shift':((meta.dom=='loc_shift')|((meta.lab==1)&(meta.dom=='train_dom')))&(meta.part=='test'),
                           'pow_shift':((meta.dom=='pow_shift')|((meta.lab==0)&(meta.dom=='train_dom')))&(meta.part=='test')}.items():
            Xe,ye,_=pairs(meta,X,np.where(mask)[0],b)
            t=time.time(); p=mdl.predict_proba(Xe)[:,1]; inf_us=(time.time()-t)/len(Xe)*1e6
            fpr=((p>=0.5)&(ye==0)).sum()/max(1,(ye==0).sum()); tpr=((p>=0.5)&(ye==1)).sum()/max(1,(ye==1).sum())
            DETRES.append(dict(band=b,model=name,domain=dname,n=len(ye),pos=int(ye.sum()),AUROC=roc_auc_score(ye,p),AP=average_precision_score(ye,p),
                               F1=f1_score(ye,p>=0.5),TPR=tpr,FPR=fpr,fit_s=fit_s,infer_us_per_channel=inf_us))
dres=pd.DataFrame(DETRES); dres.to_csv(f"{CFG['OUT']}/detector_results.csv",index=False)
print(dres.round(4).to_string())

## 5. Trace-driven closed-loop emulator
One slot = one real spectral-scan file. If the jammer is ON on logical channel c, the observation is a real malicious file of the matching power class whose target channel is rotated onto c (channel-shift emulation); otherwise a real benign file from the deployment background. The UAV link outcome uses an A2G path-loss + Rician model with the true jammer state.

In [ ]:
def a2g_pl(d,f_mhz,eta=1.0):
    return 20*np.log10(np.maximum(d,1.0))+20*np.log10(f_mhz)-27.55+eta
def overlap(band,K):
    if band=='2.4':
        g=np.zeros((K,K))
        for i in range(K):
            for j in range(K): g[i,j]=max(0.0,(20-5*abs(i-j))/20)
        return g
    return np.eye(K)
class Env:
    def __init__(s,band,meta,SCORE,TIDX,domain,jtype,seed,T):
        s.b=band; s.chs=CH[band]; s.K=len(s.chs); s.T=T; s.jtype=jtype
        s.rj=np.random.default_rng(seed); s.ro=np.random.default_rng(seed+1); s.rf=np.random.default_rng(seed+2); s.r=np.random.default_rng(seed+3)
        s.G=overlap(band,s.K); s.fc=float(np.mean(s.chs))
        test=(meta.part=='test').values
        ben_dom='loc_shift' if domain in ('loc_shift','loc_pow') else 'train_dom'
        s.ben=np.where(test&(meta.lab==0).values&(meta.dom==ben_dom).values)[0]
        mal=test&(meta.lab==1).values
        pw_tr=sorted(meta.loc[(meta.lab==1)&(meta.dom=='train_dom'),'pw'].unique())
        pw_all=sorted(meta.loc[meta.lab==1,'pw'].unique())
        s.pclasses=pw_all if domain in ('pow_shift','loc_pow') else pw_tr
        dom_ok=(meta.dom.isin(['train_dom','pow_shift'])).values if domain in ('pow_shift','loc_pow') else (meta.dom=='train_dom').values
        s.mal={p:np.where(mal&dom_ok&(meta.pw==p).values)[0] for p in s.pclasses}
        s.score=SCORE; s.tidx=TIDX
        s.jpos=np.array([1200.0,0,0]); s.Ptx=23; s.Pj=30; s.N=-95; s.gth=5.0; s.K_ric=10**(7.6/10)
        tt=np.arange(600); pr=np.array([s.Pj-a2g_pl(np.linalg.norm(s.uav(t)-s.jpos),s.fc) for t in tt])
        s.pbins=np.quantile(pr,np.linspace(0,1,len(s.pclasses)+1)[1:-1])   # weakest received power -> lowest power class
    def uav(s,t):
        a=2*np.pi*t/600; return np.array([900*np.cos(a),900*np.sin(a),120.0])
    def prx(s,t): return s.Pj-a2g_pl(np.linalg.norm(s.uav(t)-s.jpos),s.fc)
    def power_class(s):
        pr=s.prx(s.t); return s.pclasses[int(np.searchsorted(s.pbins,pr))], pr
    def reset(s):
        s.t=0; s.c=int(s.rj.integers(s.K)); s.on=True; return s.observe()
    def jam_step(s,a_prev):
        s.on = s.rj.random()<(0.95 if s.on else 0.2)
        if s.jtype=='sweep': s.c=(s.c+1)%s.K if s.t%3==0 else s.c
        elif s.jtype=='markov': s.c=s.c if s.rj.random()<0.9 else int(s.rj.integers(s.K))
        elif s.jtype=='reactive': s.c=a_prev if s.rj.random()<0.7 else s.c
    def observe(s):
        if s.on:
            p,_=s.power_class(); i=int(s.ro.choice(s.mal[p])); obs=np.roll(s.score[i],s.c-s.tidx[i])
        else:
            i=int(s.ro.choice(s.ben)); obs=s.score[i]
        s.obs=obs; return obs
    def step(s,a):
        d=np.linalg.norm(s.uav(s.t)); S=s.Ptx-a2g_pl(d,s.fc)
        h=s.rf.normal(np.sqrt(s.K_ric/(s.K_ric+1)),np.sqrt(1/(2*(s.K_ric+1))),2); fade=10*np.log10(h[0]**2+h[1]**2+1e-12)
        J=10**(s.prx(s.t)/10)*s.G[a,s.c] if s.on else 0.0
        sinr=S+fade-10*np.log10(10**(s.N/10)+J)
        ack=int(sinr>=s.gth); jammed=int(s.on and s.G[a,s.c]>0)
        s.t+=1; s.jam_step(a); obs=s.observe()
        return obs,ack,jammed

def band_scores(meta,X,det,band):
    K=len(CH[band]); idx=np.where(meta.part=='test')[0]
    SC=np.full((len(meta),K),np.nan); SC[idx]=det.predict_proba(X[idx].reshape(-1,X.shape[2]))[:,1].reshape(len(idx),K)
    TI=np.full(len(meta),-1)
    for i in idx:
        if meta.lab.iat[i]==1: TI[i]=CH[band].index(int(meta.tch.iat[i]))
    return SC,TI

## 6. Policies: oracle state (as assumed in the literature), no sensing, detector-driven, DQN, and the proposed feedback-calibrated conformal sensing (FCCS)

In [ ]:
class Oracle:
    name='Oracle-state'
    def reset(s,env): s.a=0
    def act(s,env,obs):
        if env.on and env.G[s.a,env.c]>0:
            free=[k for k in range(env.K) if env.G[k,env.c]==0]; s.a=int(env.r.choice(free))
        return s.a
    def feedback(s,a,ack,obs): pass
class RandomFH:
    name='Random-FH'
    def reset(s,env): pass
    def act(s,env,obs): return int(env.r.integers(env.K))
    def feedback(s,a,ack,obs): pass
class AckBandit:
    name='ACK-only SW-UCB'
    def reset(s,env,W=50):
        s.K=env.K; s.W=W; s.buf=np.zeros((env.K,W)); s.n=np.zeros(env.K,int); s.ptr=np.zeros(env.K,int); s.t=0; s.a=0
    def act(s,env,obs):
        s.t+=1; n=np.minimum(s.n,s.W)
        mean=np.where(n>0,s.buf.sum(1)/np.maximum(n,1),0)
        v=np.where(n>0,mean+np.sqrt(0.5*np.log(min(s.t,s.W*s.K))/np.maximum(n,1)),1e9)
        s.a=int(np.argmax(v)); return s.a
    def feedback(s,a,ack,obs):
        s.buf[a,s.ptr[a]]=ack; s.ptr[a]=(s.ptr[a]+1)%s.W; s.n[a]+=1
def affected(G,jset):
    """channels whose band overlaps any channel in jset (G = spectral overlap matrix)"""
    if len(jset)==0: return np.zeros(G.shape[0],bool)
    return (G[:,list(jset)]>0).any(1)
def pick_clean(env,aff,pref):
    cand=np.where(~aff)[0]
    if len(cand)==0: return int(np.argmax(pref))
    best=cand[pref[cand]>=pref[cand].max()-1e-12]
    return int(env.r.choice(best))
class DetHard:
    name='Detector hard (0.5)'
    def reset(s,env): s.a=0
    def act(s,env,obs):
        aff=affected(env.G,np.where(obs>=0.5)[0])
        if aff[s.a]: s.a=pick_clean(env,aff,-env.G.dot(obs))
        return s.a
    def feedback(s,a,ack,obs): pass
class DetSoft:
    name='Detector soft (overlap-weighted risk)'
    def reset(s,env): s.a=0
    def act(s,env,obs):
        risk=(env.G*obs[None,:]).max(1)
        k=int(env.r.choice(np.where(risk<=risk.min()+1e-12)[0]))
        if risk[s.a]-risk[k]>0.2: s.a=k
        return s.a
    def feedback(s,a,ack,obs): pass
class FCCS:
    """Feedback-calibrated conformal sensing (proposed).
    Per-channel reference sets hold detector scores seen on slots where the UAV used that channel and got an ACK
    (free labels from the link). Conformal p-value p_k=(1+#{r>=s_k})/(n_k+1); channel k is declared jammed if p_k<alpha_t.
    alpha_t follows adaptive conformal inference: it rises after a NACK on a channel that was declared clean (miss)
    and slowly decays otherwise, tracking a target miss rate. Warm start: training-domain calibration benign scores."""
    name='FCCS (conformal)'
    def __init__(s,warm,target=0.05,gamma=0.01,cap=400): s.warm=np.asarray(warm); s.target=target; s.gamma=gamma; s.cap=cap
    def reset(s,env):
        s.K=env.K; s.Q=[list(s.warm[-s.cap:]) for _ in range(env.K)]; s.R=[np.sort(q) for q in s.Q]; s.alpha=0.05; s.a=0; s.aff=None
    def pvals(s,obs):
        return np.array([(1+len(s.R[k])-np.searchsorted(s.R[k],obs[k],side='left'))/(len(s.R[k])+1) for k in range(s.K)])
    def act(s,env,obs):
        p=s.pvals(obs); s.aff=affected(env.G,np.where(p<s.alpha)[0])
        if s.aff[s.a]: s.a=pick_clean(env,s.aff,(env.G*p[None,:]+(1-env.G)).min(1))
        return s.a
    def feedback(s,a,ack,obs):
        declared_clean=not s.aff[a]
        if ack:
            s.Q[a].append(float(obs[a])); s.Q[a]=s.Q[a][-s.cap:]; s.R[a]=np.sort(s.Q[a])   # FIFO window, sorted copy
        err=float(declared_clean and not ack)
        s.alpha=float(np.clip(s.alpha+s.gamma*(err-s.target),0.001,0.5))

class FBR:
    """Feedback-based recalibration (proposed v3). The detector ranks channels well under shift but its 0.5 threshold
    does not transfer (AUROC stays high while TPR collapses for weak jammers). FBR keeps the detector frozen and learns an
    online Platt map q = sigmoid(w*logit(s)+b) from the link's own ACK/NACK on the channel in use (label y = NACK).
    Starts at the identity map (w=1, b=0), so with no shift it behaves like the hard-threshold detector."""
    name='FBR (online Platt)'
    def __init__(s,lr=0.05): s.lr=lr
    def reset(s,env): s.w=1.0; s.b=0.0; s.a=0
    def q(s,obs):
        z=np.log(np.clip(obs,1e-6,1-1e-6)/(1-np.clip(obs,1e-6,1-1e-6))); return 1/(1+np.exp(-(s.w*z+s.b))),z
    def act(s,env,obs):
        q,_=s.q(obs); aff=affected(env.G,np.where(q>=0.5)[0])
        if aff[s.a]: s.a=pick_clean(env,aff,-env.G.dot(q))
        return s.a
    def feedback(s,a,ack,obs):
        q,z=s.q(obs); y=1.0-ack; g=y-q[a]
        s.w=float(np.clip(s.w+s.lr*g*z[a]*0.1,0.2,5.0)); s.b=float(np.clip(s.b+s.lr*g,-5,15))

In [ ]:
import torch, torch.nn as nn
torch.set_num_threads(1)
class QNet(nn.Module):
    def __init__(s,K): super().__init__(); s.f=nn.Sequential(nn.Linear(2*K,128),nn.ReLU(),nn.Linear(128,128),nn.ReLU(),nn.Linear(128,K))
    def forward(s,x): return s.f(x)
class DQNPolicy:
    def __init__(s,name,K,state_mode): s.name=name; s.K=K; s.mode=state_mode; s.q=QNet(K); s.eps=0.0
    def state(s,env,obs):
        if s.mode=='oracle': j=np.array([float(env.on and env.G[k,env.c]>0) for k in range(env.K)])
        elif s.mode=='det_hard': j=(obs>=0.5).astype(float)
        else: j=obs.astype(float)
        cur=np.zeros(env.K); cur[s.a]=1
        return np.concatenate([j,cur]).astype(np.float32)
    def reset(s,env): s.a=0; s.sync()
    def sync(s):
        W=[p.detach().numpy().copy() for p in s.q.parameters()]; s.W=W
    def qnp(s,x):
        W=s.W; h=np.maximum(W[0]@x+W[1],0); h=np.maximum(W[2]@h+W[3],0); return W[4]@h+W[5]
    def act(s,env,obs):
        st=s.state(env,obs)
        if env.r.random()<s.eps: s.a=int(env.r.integers(env.K))
        else: s.a=int(np.argmax(s.qnp(st)))
        s.last_state=st; return s.a
    def feedback(s,a,ack,obs): pass
def train_dqn(pol,make_env,episodes,T,switch_cost=0.05,seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    tgt=QNet(pol.K); tgt.load_state_dict(pol.q.state_dict()); opt=torch.optim.Adam(pol.q.parameters(),1e-3)
    buf=[]; step=0
    for ep in range(episodes):
        env=make_env(1000+ep); obs=env.reset(); pol.reset(env)
        pol.eps=max(0.05,1-ep/(0.6*episodes))
        for t in range(T):
            a_prev=pol.a; a=pol.act(env,obs); st=pol.last_state
            obs2,ack,_=env.step(a); r=ack-switch_cost*(a!=a_prev)
            st2=pol.state(env,obs2); buf.append((st,a,r,st2)); buf=buf[-50000:]; obs=obs2; step+=1
            if len(buf)>1000 and step%8==0:
                b=random.sample(buf,128); S0=torch.tensor(np.stack([x[0] for x in b])); A=torch.tensor([x[1] for x in b])
                R=torch.tensor([x[2] for x in b],dtype=torch.float32); S1=torch.tensor(np.stack([x[3] for x in b]))
                with torch.no_grad(): y=R+0.9*tgt(S1).max(1).values
                q=pol.q(S0).gather(1,A[:,None]).squeeze(1); loss=nn.functional.smooth_l1_loss(q,y)
                opt.zero_grad(); loss.backward(); opt.step()
                if step%64==0: pol.sync()
            if step%1000==0: tgt.load_state_dict(pol.q.state_dict())
    pol.eps=0.0; pol.sync(); return pol

## 7. Run closed-loop experiments

In [ ]:
def run(env,pol):
    obs=env.reset(); pol.reset(env); acks=0; jam=0; hops=0; a_prev=None; miss=0; falsehop=0
    for t in range(env.T):
        a=pol.act(env,obs)
        if a_prev is not None and a!=a_prev:
            hops+=1
            if not (env.on and env.G[a_prev,env.c]>0): falsehop+=1
        obs2,ack,jammed=env.step(a); pol.feedback(a,ack,obs)
        acks+=ack; jam+=jammed; a_prev=a; obs=obs2
    T=env.T
    return dict(PSR=acks/T,jammed_frac=jam/T,hops_per100=100*hops/T,false_hops_per100=100*falsehop/T)
from joblib import Parallel, delayed
DOMAINS=['matched','loc_shift','pow_shift','loc_pow']; JTYPES=['sweep','markov','reactive']
ROWS=[]
for b in CFG['BANDS']:
    meta,X=DATA[b]; det=DET[(b,'HistGB')]
    calb=np.where((meta.part=='cal')&(meta.dom=='train_dom')&(meta.lab==0))[0]
    warm=det.predict_proba(X[calb].reshape(-1,X.shape[2]))[:,1]
    K=len(CH[b]); SC,TI=band_scores(meta,X,det,b)
    np.savez_compressed(f"{CFG['OUT']}/scores_{b}GHz.npz",SC=SC,TI=TI,warm=warm)
    meta[['lab','grp','pw','tch','part','dom']].to_csv(f"{CFG['OUT']}/meta_{b}GHz.csv",index=False)
    mk=lambda jt,dom: (lambda sd: Env(b,meta,SC,TI,dom,jt,sd,CFG['T_SLOTS']))
    dqn={}
    for mode in ['oracle','det_soft']:
        for jt in JTYPES:
            pol=DQNPolicy(f'DQN[{mode}]',K,mode); t=time.time()
            dqn[(mode,jt)]=train_dqn(pol,mk(jt,'matched'),CFG['DQN_EPISODES'],CFG['DQN_T'])
            print(b,mode,jt,f'DQN trained {time.time()-t:.0f}s')
    pd.DataFrame(DETRES).to_csv(f"{CFG['OUT']}/detector_results.csv",index=False)
    def task(dom,jt,sd):
        torch.set_num_threads(1); out=[]
        env_seed=10000+sd
        pols=[Oracle(),RandomFH(),AckBandit(),DetHard(),DetSoft(),FCCS(warm),FBR()]
        d_or=dqn[('oracle',jt)]; d_or.name='DQN trained on oracle state'
        d_dt=dqn[('det_soft',jt)]; d_dt.name='DQN trained on detector scores'
        d_mis=DQNPolicy('DQN oracle-trained, detector-deployed',K,'det_hard'); d_mis.q=d_or.q   # the usual sim-to-real gap
        for pol in pols+[d_or,d_dt,d_mis]:
            env=Env(b,meta,SC,TI,dom,jt,env_seed,CFG['T_SLOTS'])
            r=run(env,pol); r.update(band=b,domain=dom,jammer=jt,seed=sd,policy=pol.name); out.append(r)
        return out
    jobs=[(dom,jt,sd) for dom in DOMAINS for jt in JTYPES for sd in CFG['SEEDS']]
    t=time.time()
    res=Parallel(n_jobs=-1,verbose=5)(delayed(task)(*j) for j in jobs)
    for r in res: ROWS.extend(r)
    pd.DataFrame(ROWS).to_csv(f"{CFG['OUT']}/closed_loop_raw.csv",index=False)   # checkpoint after each band
    print(b,'closed loop done',f'{time.time()-t:.0f}s, total {time.time()-T0:.0f}s')
CL=pd.DataFrame(ROWS); CL.to_csv(f"{CFG['OUT']}/closed_loop_raw.csv",index=False)
summ=CL.groupby(['band','domain','jammer','policy'])[['PSR','jammed_frac','hops_per100','false_hops_per100']].mean().round(4)
summ.to_csv(f"{CFG['OUT']}/closed_loop_summary.csv"); print(summ.to_string())

## 8. Paired statistics (FCCS vs every baseline; Wilcoxon over seeds x jammer types) and figures

In [ ]:
from scipy.stats import wilcoxon
stats=[]
for (b,dom),g in CL.groupby(['band','domain']):
    piv=g.pivot_table(index=['jammer','seed'],columns='policy',values='PSR')
    for p in piv.columns:
        if p=='FBR (proposed)': continue
        d=piv['FBR (proposed)']-piv[p]
        try: pv=wilcoxon(d).pvalue
        except Exception: pv=np.nan
        stats.append(dict(band=b,domain=dom,vs=p,mean_diff_pp=100*d.mean(),wins=int((d>0).sum()),n=len(d),p=pv,gap_to_oracle_pp=100*(piv['Oracle-state']-piv[p]).mean()))
ST=pd.DataFrame(stats); ST.to_csv(f"{CFG['OUT']}/paired_stats.csv",index=False); print(ST.round(4).to_string())

In [ ]:
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
for b in CFG['BANDS']:
    g=CL[CL.band==b].groupby(['domain','policy']).PSR.mean().unstack(0)[DOMAINS]
    ax=g.plot.barh(figsize=(8,5)); ax.set_xlabel('Packet success rate'); ax.set_title(f'{b} GHz: closed-loop PSR by domain'); plt.tight_layout()
    plt.savefig(f"{CFG['OUT']}/fig_psr_{b}GHz.png",dpi=200); plt.close()
json.dump(dict(cfg=CFG,runtime_s=time.time()-T0,n_files={b:int(len(DATA[b][0])) for b in CFG['BANDS']}),open(f"{CFG['OUT']}/run_info.json",'w'),indent=1)
print('DONE',f'{time.time()-T0:.0f}s')